In [1]:
# ---------------------------------------------------------------
# Imports and Configuration
# ---------------------------------------------------------------
from pathlib import Path
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path(r"C:\Users\tjoldham\Documents\Classes\MADS\Programming\DSCI8413\LBNL_FDD_Data_Sets\Chiller_Plant\LBNL_FDD_Dataset_Chiller_Plant")
# produced at the end of the assessment notebook
INTERMEDIATE_PATH = Path(r"C:\Users\tjoldham\Documents\Classes\MADS\Programming\DSCI8413\raw_combined.parquet")  


In [2]:
# ---------------------------------------------------------------
# Load combined data — prefer the saved intermediate from the
# assessment notebook; fall back to rebuilding from raw CSVs if
# this notebook is run standalone
# ---------------------------------------------------------------
def parse_fault_label(filename: str) -> dict:
    """Extract fault_type and fault_intensity from LBNL chiller plant filenames."""
    stem = filename.replace(".csv", "")
    if stem == "ChillerPlant":
        return {"fault_type": "fault_free", "fault_intensity": None}
    m = re.match(r"ChillerPlant_(.+)_(-?\d+)$", stem)
    if m:
        return {"fault_type": m.group(1), "fault_intensity": m.group(2)}
    return {"fault_type": stem.replace("ChillerPlant_", ""), "fault_intensity": "faulted"}

if INTERMEDIATE_PATH.exists():
    data = pd.read_parquet(INTERMEDIATE_PATH)
    print(f"Loaded intermediate file: {data.shape}")
else:
    csv_files = sorted(DATA_DIR.glob("*.csv"))
    label_map = {f.name: parse_fault_label(f.name) for f in csv_files}

    dfs = []
    for f in csv_files:
        df = pd.read_csv(f)
        df["source_file"] = f.name
        df["fault_type"] = label_map[f.name]["fault_type"]
        df["fault_intensity"] = label_map[f.name]["fault_intensity"]
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    print(f"Rebuilt from raw CSVs: {data.shape}")
    del dfs

Loaded intermediate file: (12612960, 83)


In [3]:
# ---------------------------------------------------------------
# Transformation 1: Datetime conversion
# Raw Datetime loads as string; required before any chronological
# operation (sorting, gap detection, month extraction)
# ---------------------------------------------------------------
data["Datetime"] = pd.to_datetime(data["Datetime"], errors="coerce")
assert data["Datetime"].isna().sum() == 0, "Unparseable timestamps found — investigate before proceeding"
print(f"Datetime dtype after conversion: {data['Datetime'].dtype}")

Datetime dtype after conversion: datetime64[us]


In [4]:
# ---------------------------------------------------------------
# Transformation 2: Split fault_intensity into unit-consistent
# columns. Raw values mix signed °C offsets, zero-padded percentages,
# and a non-numeric flag ("faulted") under one column — treating
# them as one numeric scale would compare incompatible units.
# ---------------------------------------------------------------
bias_types = {"chiller_bias", "coolingtower_bias"}
pct_types = {"bypass_leakage", "bypass_stuck", "chiller_fouling",
             "coolingtower_fouling", "secondary_chilled_water_pressure_bias"}

def to_signed_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

data["intensity_degC"] = np.where(
    data["fault_type"].isin(bias_types),
    data["fault_intensity"].map(to_signed_float),
    np.nan,
)
data["intensity_pct"] = np.where(
    data["fault_type"].isin(pct_types),
    data["fault_intensity"].map(to_signed_float),
    np.nan,
)

# Verify the split covers every fault type appropriately
check = data[["fault_type", "fault_intensity", "intensity_degC", "intensity_pct"]].drop_duplicates()
print(check.sort_values("fault_type").to_string(index=False))

                           fault_type fault_intensity  intensity_degC  intensity_pct
                       bypass_leakage             025             NaN           25.0
                       bypass_leakage             050             NaN           50.0
                       bypass_leakage             075             NaN           75.0
                         bypass_stuck             050             NaN           50.0
                         bypass_stuck             075             NaN           75.0
                         chiller_bias              -1            -1.0            NaN
                         chiller_bias              -2            -2.0            NaN
                         chiller_bias               1             1.0            NaN
                         chiller_bias               2             2.0            NaN
                      chiller_fouling             065             NaN           65.0
                      chiller_fouling             095            

In [5]:
# ---------------------------------------------------------------
# Transformation 3: Flag zero-variance columns
# These carry no discriminative information for modeling and can
# cause numerical issues during feature standardization (division
# by a zero standard deviation). Retained in `data` for documentation,
# but excluded from the modeling feature set defined below.
# ---------------------------------------------------------------
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("intensity_degC", "intensity_pct")]

stds = data[numeric_cols].std()
zero_variance_cols = stds[stds < 1e-6].index.tolist()
print(f"Zero-variance columns flagged for exclusion: {zero_variance_cols}")

Zero-variance columns flagged for exclusion: ['CHL_STA_1', 'CT_STA_1', 'CWL_SEC_DPSPT', 'CWL_SEC_PM_STA_1']


In [6]:
# ---------------------------------------------------------------
# Define the modeling feature set
# Excludes zero-variance columns and label/metadata columns.
# This is the column list downstream feature engineering and
# model training should reference, rather than `data.columns`
# directly, to keep the exclusion decision in one place.
# ---------------------------------------------------------------
non_feature_cols = zero_variance_cols + [
    "Datetime", "source_file", "fault_type", "fault_intensity"
]

model_feature_cols = [c for c in numeric_cols if c not in non_feature_cols]
print(f"Modeling feature set: {len(model_feature_cols)} columns")
print(model_feature_cols)

Modeling feature set: 73 columns
['CDWL_CW_FLOW', 'CDWL_PM_POW_1', 'CDWL_PM_POW_2', 'CDWL_PM_POW_3', 'CDWL_RW_TEMP', 'CDWL_SW_TEMP', 'CHL_CD_FLOW_1', 'CHL_CD_FLOW_2', 'CHL_CD_FLOW_3', 'CHL_COMP_SPD_CTRL_1', 'CHL_COMP_SPD_CTRL_2', 'CHL_COMP_SPD_CTRL_3', 'CHL_CW_FLOW_1', 'CHL_CW_FLOW_2', 'CHL_CW_FLOW_3', 'CHL_POW_1', 'CHL_POW_2', 'CHL_POW_3', 'CHL_SWCD_TEMP_1', 'CHL_SWCD_TEMP_2', 'CHL_SWCD_TEMP_3', 'CHL_RW_TEMP_1', 'CHL_RW_TEMP_2', 'CHL_RW_TEMP_3', 'CHL_STA_2', 'CHL_STA_3', 'CHL_RWCD_TEMP_1', 'CHL_RWCD_TEMP_2', 'CHL_RWCD_TEMP_3', 'CHL_SW_TEMP_1', 'CHL_SW_TEMP_2', 'CHL_SW_TEMP_3', 'CT_FAN_SPD_1', 'CT_FAN_SPD_2', 'CT_FAN_SPD_3', 'CT_FAN_SPD_CTRL_1', 'CT_FAN_SPD_CTRL_2', 'CT_FAN_SPD_CTRL_3', 'CT_FLOW_1', 'CT_FLOW_2', 'CT_FLOW_3', 'CT_POW_1', 'CT_POW_2', 'CT_POW_3', 'CT_RW_TEMP_1', 'CT_RW_TEMP_2', 'CT_RW_TEMP_3', 'CT_STA_2', 'CT_STA_3', 'CT_SW_TEMPSPT', 'CT_SW_TEMP_1', 'CT_SW_TEMP_2', 'CT_SW_TEMP_3', 'CWL_PRI_CW_FLOW', 'CWL_PRI_PM_POW_1', 'CWL_PRI_PM_POW_2', 'CWL_PRI_PM_POW_3', 'CWL_PRI_RW_T

In [7]:
# ---------------------------------------------------------------
# Outlier handling — documented decision, no rows removed
# Global IQR flags were investigated in the assessment notebook and
# attributed to genuine multimodality across staging/load regimes,
# not to corrupted values. No filtering is applied here. This cell
# exists to make that decision explicit and auditable in the
# cleaning notebook, rather than silently absent.
# ---------------------------------------------------------------
print(
    "Outlier handling decision: retained all records flagged by global IQR "
    "screening. Elevated flag rates (up to ~42-47% on select temperature/"
    "flow columns) were attributed to regime multimodality (chiller-staging "
    "state and outdoor-air-driven load), not measurement error. No rows "
    "removed on this basis. Regime-conditional treatment is recommended for "
    "any future anomaly-detection step; see assessment notebook for the "
    "supporting investigation."
)

Outlier handling decision: retained all records flagged by global IQR screening. Elevated flag rates (up to ~42-47% on select temperature/flow columns) were attributed to regime multimodality (chiller-staging state and outdoor-air-driven load), not measurement error. No rows removed on this basis. Regime-conditional treatment is recommended for any future anomaly-detection step; see assessment notebook for the supporting investigation.


In [8]:
# ---------------------------------------------------------------
# Undocumented scenario — confirmed legitimate, no action needed
# chiller_fouling (65%, 95%) was flagged during assessment as absent
# from the source documentation. Confirmed as a scenario added after
# the documentation was written. No special-casing required — its
# filename and intensity convention already match the documented
# coolingtower_fouling pattern and were handled correctly by the
# standard parsing/splitting logic above.
# ---------------------------------------------------------------
assert "chiller_fouling" in data["fault_type"].unique()
print("chiller_fouling confirmed present and correctly labeled — no cleaning action required.")

chiller_fouling confirmed present and correctly labeled — no cleaning action required.


In [9]:
# ---------------------------------------------------------------
# Deferred items — documented, not resolved in this notebook
# Class imbalance: left as-is; correction (stratified split or
# resampling) belongs at the train/test-split stage, not here.
# Potential leakage (CT_RW_TEMP_2/3, CT_SW_TEMP_2/3): left as-is;
# requires further verification before any feature-level decision.
# ---------------------------------------------------------------
print("Deferred to modeling stage: class imbalance handling (stratified split/resampling).")
print("Deferred pending verification: potential leakage in CT_RW_TEMP_2/3, CT_SW_TEMP_2/3.")

Deferred to modeling stage: class imbalance handling (stratified split/resampling).
Deferred pending verification: potential leakage in CT_RW_TEMP_2/3, CT_SW_TEMP_2/3.


In [10]:
# ---------------------------------------------------------------
# Save cleaned/transformed dataset for downstream modeling notebook
# ---------------------------------------------------------------
OUTPUT_PATH = Path("cleaned_transformed.parquet")
data.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset: {data.shape} -> {OUTPUT_PATH}")
print(f"Modeling feature set saved separately for reference: {len(model_feature_cols)} columns")

import json
with open("model_feature_cols.json", "w") as f:
    json.dump(model_feature_cols, f, indent=2)

Saved cleaned dataset: (12612960, 83) -> cleaned_transformed.parquet
Modeling feature set saved separately for reference: 73 columns
